# Distribution Metrics for Test vs Generated Samples

Ноутбук сравнивает распределение реальных полей из `test` и `1000` сгенерированных samples.

Что считается:
- пиксельные гистограммы по каждому каналу
- квантили по каждому каналу
- распределения image-level mean по каждому каналу
- распределения image-level std по каждому каналу
- компактная таблица с базовыми summary statistics


In [ ]:
import json
import os
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if platform.system() == 'Darwin':
    REPO_DIR = '/Users/amir/sciml/diffusion_data_assimilation'
    DATA_ROOT = '/Users/amir/sciml/sea_ice_data'
else:
    REPO_DIR = os.getcwd()
    DATA_ROOT = '/mnt/sciml/a.sadreev/sea_ice_data'

os.chdir(REPO_DIR)

WORK_DIR = '/tmp/inception_fid_test_eval'
REAL_DIR = os.path.join(WORK_DIR, 'real_test_subset')
FAKE_DIR = os.path.join(WORK_DIR, 'fake_test_subset_concat')
MASK_PATH = os.path.join(DATA_ROOT, 'mask_padding.npy')

CHANNEL_NAMES = ['channel_0', 'channel_1']
PIXEL_SAMPLE_SIZE = 200_000
HIST_BINS = 80
QUANTILES = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
RNG_SEED = 1234

print('repo_dir        =', REPO_DIR)
print('real_dir        =', REAL_DIR)
print('fake_dir        =', FAKE_DIR)
print('mask_path       =', MASK_PATH)
print('pixel_sample_size =', PIXEL_SAMPLE_SIZE)


In [ ]:
def to_channel_first(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.ndim == 2:
        return arr[None, ...].astype(np.float32, copy=False)
    if arr.ndim != 3:
        raise ValueError(f'Unsupported array shape {arr.shape}')
    if arr.shape[0] in (1, 2, 3):
        out = arr
    elif arr.shape[-1] in (1, 2, 3):
        out = np.moveaxis(arr, -1, 0)
    else:
        raise ValueError(f'Unsupported array shape {arr.shape}')
    return out.astype(np.float32, copy=False)


real_files = sorted([p.name for p in Path(REAL_DIR).glob('*.npy')])
fake_files = sorted([p.name for p in Path(FAKE_DIR).glob('*.npy')])
shared_files = sorted(set(real_files) & set(fake_files))
assert shared_files, 'No shared .npy files between real and fake folders'

valid_mask = np.load(MASK_PATH).astype(bool)
assert valid_mask.ndim == 2, f'Expected 2D mask, got {valid_mask.shape}'

sample = to_channel_first(np.load(os.path.join(REAL_DIR, shared_files[0])))
n_channels = sample.shape[0]
assert len(CHANNEL_NAMES) >= n_channels, 'CHANNEL_NAMES must cover all channels'

print('shared_files    =', len(shared_files))
print('sample shape    =', sample.shape)
print('valid pixels    =', int(valid_mask.sum()))
print('first 5 files   =', shared_files[:5])


In [ ]:
rng = np.random.default_rng(RNG_SEED)

pixel_values_real = [[] for _ in range(n_channels)]
pixel_values_fake = [[] for _ in range(n_channels)]
image_means_real = [[] for _ in range(n_channels)]
image_means_fake = [[] for _ in range(n_channels)]
image_stds_real = [[] for _ in range(n_channels)]
image_stds_fake = [[] for _ in range(n_channels)]

for name in shared_files:
    real_arr = to_channel_first(np.load(os.path.join(REAL_DIR, name)))
    fake_arr = to_channel_first(np.load(os.path.join(FAKE_DIR, name)))

    for ch in range(n_channels):
        real_vals = real_arr[ch][valid_mask]
        fake_vals = fake_arr[ch][valid_mask]

        pixel_values_real[ch].append(real_vals)
        pixel_values_fake[ch].append(fake_vals)
        image_means_real[ch].append(float(real_vals.mean()))
        image_means_fake[ch].append(float(fake_vals.mean()))
        image_stds_real[ch].append(float(real_vals.std()))
        image_stds_fake[ch].append(float(fake_vals.std()))

pixel_values_real = [np.concatenate(v) for v in pixel_values_real]
pixel_values_fake = [np.concatenate(v) for v in pixel_values_fake]
image_means_real = [np.asarray(v, dtype=np.float64) for v in image_means_real]
image_means_fake = [np.asarray(v, dtype=np.float64) for v in image_means_fake]
image_stds_real = [np.asarray(v, dtype=np.float64) for v in image_stds_real]
image_stds_fake = [np.asarray(v, dtype=np.float64) for v in image_stds_fake]

pixel_plot_real = []
pixel_plot_fake = []
for ch in range(n_channels):
    real_vals = pixel_values_real[ch]
    fake_vals = pixel_values_fake[ch]
    real_idx = rng.choice(real_vals.shape[0], size=min(PIXEL_SAMPLE_SIZE, real_vals.shape[0]), replace=False)
    fake_idx = rng.choice(fake_vals.shape[0], size=min(PIXEL_SAMPLE_SIZE, fake_vals.shape[0]), replace=False)
    pixel_plot_real.append(real_vals[real_idx])
    pixel_plot_fake.append(fake_vals[fake_idx])

print('total pixel samples per channel (real) =', [int(v.shape[0]) for v in pixel_values_real])
print('total pixel samples per channel (fake) =', [int(v.shape[0]) for v in pixel_values_fake])


In [ ]:
quantile_rows = []
summary_rows = []

for ch in range(n_channels):
    q_real = np.quantile(pixel_values_real[ch], QUANTILES)
    q_fake = np.quantile(pixel_values_fake[ch], QUANTILES)
    for q, rv, fv in zip(QUANTILES, q_real, q_fake):
        quantile_rows.append({
            'channel': CHANNEL_NAMES[ch],
            'quantile': q,
            'real': float(rv),
            'fake': float(fv),
            'fake_minus_real': float(fv - rv),
        })

    summary_rows.append({
        'channel': CHANNEL_NAMES[ch],
        'pixel_mean_real': float(pixel_values_real[ch].mean()),
        'pixel_mean_fake': float(pixel_values_fake[ch].mean()),
        'pixel_std_real': float(pixel_values_real[ch].std()),
        'pixel_std_fake': float(pixel_values_fake[ch].std()),
        'image_mean_mean_real': float(image_means_real[ch].mean()),
        'image_mean_mean_fake': float(image_means_fake[ch].mean()),
        'image_mean_std_real': float(image_means_real[ch].std()),
        'image_mean_std_fake': float(image_means_fake[ch].std()),
        'image_std_mean_real': float(image_stds_real[ch].mean()),
        'image_std_mean_fake': float(image_stds_fake[ch].mean()),
    })

quantile_df = pd.DataFrame(quantile_rows)
summary_df = pd.DataFrame(summary_rows)

display(summary_df)
display(quantile_df)


In [ ]:
fig, axes = plt.subplots(n_channels, 3, figsize=(16, 5 * n_channels))
axes = np.atleast_2d(axes)

for ch in range(n_channels):
    ax = axes[ch, 0]
    data_min = min(pixel_plot_real[ch].min(), pixel_plot_fake[ch].min())
    data_max = max(pixel_plot_real[ch].max(), pixel_plot_fake[ch].max())
    bins = np.linspace(data_min, data_max, HIST_BINS + 1)
    ax.hist(pixel_plot_real[ch], bins=bins, density=True, alpha=0.5, label='test', color='tab:blue')
    ax.hist(pixel_plot_fake[ch], bins=bins, density=True, alpha=0.5, label='generated', color='tab:orange')
    ax.set_title(f'{CHANNEL_NAMES[ch]}: pixel histogram')
    ax.set_xlabel('value')
    ax.set_ylabel('density')
    ax.legend()

    ax = axes[ch, 1]
    bins = np.linspace(
        min(image_means_real[ch].min(), image_means_fake[ch].min()),
        max(image_means_real[ch].max(), image_means_fake[ch].max()),
        HIST_BINS + 1,
    )
    ax.hist(image_means_real[ch], bins=bins, density=True, alpha=0.5, label='test', color='tab:blue')
    ax.hist(image_means_fake[ch], bins=bins, density=True, alpha=0.5, label='generated', color='tab:orange')
    ax.set_title(f'{CHANNEL_NAMES[ch]}: image mean histogram')
    ax.set_xlabel('image mean')
    ax.set_ylabel('density')
    ax.legend()

    ax = axes[ch, 2]
    bins = np.linspace(
        min(image_stds_real[ch].min(), image_stds_fake[ch].min()),
        max(image_stds_real[ch].max(), image_stds_fake[ch].max()),
        HIST_BINS + 1,
    )
    ax.hist(image_stds_real[ch], bins=bins, density=True, alpha=0.5, label='test', color='tab:blue')
    ax.hist(image_stds_fake[ch], bins=bins, density=True, alpha=0.5, label='generated', color='tab:orange')
    ax.set_title(f'{CHANNEL_NAMES[ch]}: image std histogram')
    ax.set_xlabel('image std')
    ax.set_ylabel('density')
    ax.legend()

plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, n_channels, figsize=(6 * n_channels, 4))
axes = np.atleast_1d(axes)

for ch in range(n_channels):
    channel_quantiles = quantile_df[quantile_df['channel'] == CHANNEL_NAMES[ch]]
    axes[ch].plot(channel_quantiles['quantile'], channel_quantiles['real'], marker='o', label='test')
    axes[ch].plot(channel_quantiles['quantile'], channel_quantiles['fake'], marker='o', label='generated')
    axes[ch].set_title(f'{CHANNEL_NAMES[ch]}: quantile curve')
    axes[ch].set_xlabel('quantile')
    axes[ch].set_ylabel('value')
    axes[ch].grid(alpha=0.3)
    axes[ch].legend()

plt.tight_layout()
